In [1]:
from qiskit_metal import designs, MetalGUI, Dict
import numpy as np

design = designs.DesignPlanar({}, overwrite_enabled= True)
gui = MetalGUI(design)

In [2]:
from qiskit_metal.qlibrary.tlines.meandered import RouteMeander
from qiskit_metal.qlibrary.lumped.cap_n_interdigital import CapNInterdigital

In [5]:
try:
    cap.delete()
except NameError : pass

cap = CapNInterdigital(design, "cap")
gui.rebuild()
gui.autoscale()

In [6]:
cap.pin_names

{'north_end', 'south_end'}

In [25]:
from collections import OrderedDict

jogs = OrderedDict()
jogs[0] = ["L", "1000um"]
try:
    inductor.delete()
except NameError: pass

inductor = RouteMeander(design, "inductor",
    Dict(
        total_length = "5mm",
        trace_width = "cpw_width",
        fillet = "39.9um",
        pin_inputs = Dict(
            start_pin = Dict(component="cap", pin="north_end"),
            end_pin = Dict(component="cap", pin="south_end"),
        ),
        lead = Dict(start_straight="200um", end_straight="200um"),
        meander = Dict(spacing="80um", asymmetry="850um"),
    ))

gui.rebuild()
gui.autoscale()

In [3]:
import matplotlib.pyplot as plt 
from scipy.optimize import brentq
import scienceplots
import pint
plt.style.use(['science', 'no-latex', 'notebook'])
from scipy.special import ellipk


In [45]:
ureg = pint.UnitRegistry()
mu_0 = ureg.Quantity(1, ureg.mu_0).to('H/m')

def required_inductance(f0: pint.Quantity, C: pint.Quantity):
    """L needed for a given resonance frequency and capacitance: f0 = 1/(2*pi*sqrt(LC))"""
    omega0 = 2.0 * np.pi * f0.to("Hz")
    L = 1.0 / (omega0**2 * C.to("F"))
    return L.to("henry")

def line_inductance_isolated(l: pint.Quantity, w: pint.Quantity, t: pint.Quantity):
    l = l.to("um")
    w = w.to("um")
    t = t.to("um")
    
    arg_1 = (2.0 * l / (w +t)).magnitude
    arg_2 = 0.2235 * ((w + t) / l).magnitude
    L = mu_0 * l * (np.log(arg_1) + 0.5 + arg_2) / (2.0 * np.pi)
    return L.to("henry")

def line_inductance_cpw(l: pint.Quantity, w: pint.Quantity, g: pint.Quantity):
    """CPW center-conductor inductance via conformal mapping.

    w: center trace width
    g: gap to each coplanar ground plane (symmetric, infinite ground assumed)
    """
    w = w.to("um").magnitude
    g = g.to("um").magnitude

    k = w / (w + 2.0 * g)
    kp = np.sqrt(1.0 - k**2)

    # scipy's ellipk takes the parameter m = k^2, not the modulus k
    Kk = ellipk(k**2)
    Kkp = ellipk(kp**2)

    L_prime = mu_0 * 0.25 * (Kkp / Kk)  # per unit length
    L = L_prime * l.to("um")
    return L.to("henry")

def length_for_frequency_isolated(f0: pint.Quantity, C: pint.Quantity,
                                   w: pint.Quantity, t: pint.Quantity,
                                   l_bracket=(1.0, 1e5)):
    """Invert the isolated-wire formula numerically (transcendental in l).

    l_bracket: search range in um for the root finder — widen if it fails to bracket.
    """
    L_target = required_inductance(f0, C).to("henry").magnitude

    def residual(l_um):
        l = ureg.Quantity(l_um, "um")
        return line_inductance_isolated(l, w, t).to("henry").magnitude - L_target

    l_um = brentq(residual, *l_bracket)
    return ureg.Quantity(l_um, "um")

def length_for_frequency_cpw(f0: pint.Quantity, C: pint.Quantity,
                              w: pint.Quantity, g: pint.Quantity):
    """CPW inductance per unit length is constant, so this is a direct solve, not a root-find."""
    L_target = required_inductance(f0, C)

    k = w.to("um").magnitude / (w.to("um").magnitude + 2.0 * g.to("um").magnitude)
    kp = np.sqrt(1.0 - k**2)
    Kk = ellipk(k**2)
    Kkp = ellipk(kp**2)
    L_prime = mu_0 * 0.25 * (Kkp / Kk)

    l = (L_target / L_prime).to("um")
    return l


In [56]:
f0 = ureg.Quantity(8.0, "GHz")
C = ureg.Quantity(200, "fF")
w = ureg.Quantity(10, "um")
t = ureg.Quantity(100, "nm")
g = ureg.Quantity(100, "um")


In [57]:
l_isolated = length_for_frequency_isolated(f0, C, w, t)
l_cpw = length_for_frequency_cpw(f0, C, w, g)

print(f"L target: {required_inductance(f0, C).to('nH'):.4f}")
print(f"Isolated-wire length: {l_isolated:.2f}")
print(f"CPW length:           {l_cpw:.2f}")

L target: 1.9789 nanohenry
Isolated-wire length: 1583.32 micrometer
CPW length:           2233.43 micrometer


In [26]:
import geopandas as gpd
import pandas as pd

all_bounds = []
for table_name, table in design.qgeometry.tables.items():
    if len(table) == 0:
        continue
    b = table.total_bounds  # [minx, miny, maxx, maxy], in mm
    print(f"{table_name:>8}: X=[{b[0]:.4f},{b[2]:.4f}]  Y=[{b[1]:.4f},{b[3]:.4f}]")
    all_bounds.append(b)

import numpy as np
all_bounds = np.array(all_bounds)
print(f"\nOverall: X=[{all_bounds[:,0].min():.4f},{all_bounds[:,2].max():.4f}]  "
      f"Y=[{all_bounds[:,1].min():.4f},{all_bounds[:,3].max():.4f}]")

    path: X=[0.0000,1.1094]  Y=[-0.3460,0.2000]
    poly: X=[-0.0430,0.0430]  Y=[-0.1020,-0.0440]

Overall: X=[-0.0430,1.1094]  Y=[-0.3460,0.2000]
